# 机器学习算法

## 机器学习与基础数学

Q1 线性回归(梯度下降 + L2)
目标函数（L2正则化线性回归 / 岭回归）
$$ \frac{1}{n} \|Xw - y\|^2 + \lambda \|w\|^2 $$

权重 w 的梯度
$$ \frac{2}{n} X^T (Xw - y) + 2\lambda w $$

补充：偏置 b 的梯度（不含正则化）
$$ \nabla_b = \frac{2}{n} \sum_{i=1}^{n} (x_i^T w + b - y_i) = 2 \cdot \text{mean}(err) $$

In [27]:
import numpy as np

def linear_reg_gd(X, y, lr = 0.05, wd=1e-3, epochs=1000):
    n, d = X.shape
    w = np.zeros(d)
    b = 0.0
    
    for _ in range(epochs):
        pred = X @ w + b # 前向传播
        err = pred - y # 计算误差
        gw = 2 * (X.T @ err) / n + 2 * wd * w # 计算权重 w 的梯度
        gb = 2 * err.mean() # 计算偏置 b 的梯度
        
        # 梯度下降更新
        w -= lr * gw
        b -= lr * gb

    return w, b

普通最小二乘法（OLS）通过令梯度为零可直接求得闭式解：
$$ w_{OLS} = (X^T X)^{-1} X^T y $$

然而，该闭式解成立的前提是 $X^T X$ 必须可逆。在实际数据中，该矩阵常常不可逆（奇异），主要原因包括：特征维度远高于样本量（如 $100$ 个样本对应 $1000$ 个特征）导致矩阵非满秩；或特征间存在强多重共线性（如身高与脚长）导致列向量线性相关。这会造成计算机无法求逆而报错，或解出的权重值趋向无穷大，使模型完全失效。

为此，岭回归通过在矩阵对角线上施加一个正数扰动 $\lambda I$（其中 $\lambda > 0$），将闭式解改造为：
$$ w = (X^T X + \lambda I)^{-1} X^T y $$

其数学本质是：无论 $X^T X$ 多么病态，新矩阵的所有特征值都将变为 $\mu_i + \lambda$（严格大于零），从而确保其成为正定矩阵，始终可逆，从根本上规避了计算崩溃的风险。

Q2 手写线性回归(最小二乘闭式解)

In [28]:
import numpy as np

def linear_reg_closed_form(X, y, lam=1e-6):
    d = X.shape[1]
    A = X.T @ X + lam * np.eye(d) # 正则化项
    b = X.T @ y
    w = np.linalg.solve(A, b) # 求解线性方程组
    return w

逻辑回归（Logistic Regression），用于二分类问题（比如判断0/1）。

逻辑回归的梯度表达式 $\frac{1}{n} X^T (p - y)$ 源于其采用交叉熵损失函数：
$$ L = -\frac{1}{n} \sum_{i=1}^n \left[ y_i \log(p_i) + (1-y_i) \log(1-p_i) \right] $$
其中 $p_i = \text{sigmoid}(z_i)$，$z_i = x_i^T w + b$。对 $w$ 求导时，$\text{sigmoid}$ 的导数性质 $\sigma'(z) = \sigma(z)(1-\sigma(z))$ 恰好消去了分母项，最终简化为 $X^T(p-y)/n$，系数为 1。相比之下，线性回归的均方误差求导会保留系数 2，且逻辑回归的梯度中不含正则化项。

逻辑回归不存在闭式解。令梯度为零得到的方程：
$$ X^T(\text{sigmoid}(Xw) - y) = 0 $$
由于 $\text{sigmoid}$ 是非线性函数，该方程无法通过矩阵求逆等线性代数操作直接解出 $w$ 的解析表达式。因此逻辑回归只能依赖梯度下降等迭代算法逼近最优解，这凸显了在许多实际问题中闭式解的不可行性——非线性因素普遍存在时，迭代求解是唯一路径。

Q3 手写 Logistic Regression(二分类)

In [29]:
import numpy as np

def sigmoid(x):
    return 1.0 / (1.0 + np.exp(-np.clip(x, -100, 100))) # 防止溢出

def logistic_reg_train(X, y, lr = 0.1, epochs=500):
    n, d = X.shape
    w = np.zeros(d)
    b = 0.0
    
    for _ in range(epochs):
        z = X @ w + b # 前向传播
        p = sigmoid(z) # 计算预测概率
        err = (p - y).mean() # 计算误差
        gw = X.T @ (p - y) / n # 计算权重 w 的梯度
        gb = (p - y).mean() # 计算偏置 b 的梯度
        w -= lr * gw
        b -= lr * gb

    return w, b


Softmax 回归（Softmax Regression），是逻辑回归在多分类问题（类别数 > 2）上的推广。比如手写数字识别（0~9共10类）。

Q4 手写 Softmax Regression(多分类)

In [30]:
import numpy as np

def softmax(z):
    z = z - z.max(axis=1, keepdims=True) # 防止溢出
    exp_z = np.exp(z)
    return exp_z / exp_z.sum(axis=1, keepdims=True)

def softmax_reg_train(X, y, num_classes, lr=0.1, epochs=300):
    n, d = X.shape
    W = np.zeros((d, num_classes))
    Y = np.eye(num_classes)[y] # 将标签转换为 one-hot 编码
    
    for _ in range(epochs):
        P = softmax(X @ W) # 前向传播
        gW = X.T @ (P - Y) / n # 计算权重 W 的梯度
        W -= lr * gW # 更新权重
    
    return W

Q5 手写 K-Means(含空簇重置)

In [31]:
import numpy as np

def kmeans(X, k, iters=100):
    n = len(X)
    centers = X[np.random.choice(n, k, replace=False)].copy() # 随机初始化质心
    for _ in range(iters):
        dist = ((X[:, None, :] - centers[None, :, :]) ** 2).sum(-1) # 计算每个点到质心的距离
        label = dist.argmin(axis=1) # 分配标签
        new_centers = centers.copy()
        for j in range(k):
            idx = np.where(label == j)[0]
            if len(idx) == 0:
                far = dist.min(axis=1).argmax() # 找到距离最远的点
                new_centers[j] = X[far] # 将其作为新的质心
            else:
                new_centers[j] = X[idx].mean(axis=0) # 更新质心为簇内点的均值
        if np.allclose(new_centers, centers): # 如果质心没有变化，停止迭代
            break
        centers = new_centers # 更新质心
    return centers, label

Q6 KNN(支持批量预测)

In [32]:
import numpy as np

def knn_predict(X_train, y_train, X_query, k=5):
    pred = []
    for q in X_query:
        dist = ((X_train - q) ** 2).sum(axis=1) # 计算查询点到训练点的距离
        idx = dist.argsort()[:k] # 找到最近的 k 个点的索引
        pred.append(np.bincount(y_train[idx]).argmax()) # 投票
        
    return np.array(pred)

Q7 PCA(协方差矩阵 + 特征分解)

In [33]:
import numpy as np

def pca(X, k):
    Xc = X - X.mean(axis=0, keepdims=True) # 中心化
    cov = Xc.T @ Xc / (len(Xc) - 1) # 计算协方差矩阵
    vals, vecs = np.linalg.eigh(cov) # 特征分解
    idx = np.argsort(vals)[::-1][:k] # 选择前 k 个
    W = vecs[:, idx] # 选择对应的特征向量
    Z = Xc @ W # 返回降维后的数据
    return Z, W, vals[idx]

Q8 MLE 高斯分布参数估计

In [34]:
import numpy as np

def gaussian_mle(x):
    mu = x.mean() # 均值
    var = ((x - mu) ** 2).mean()
    return mu, var

Q9 KL 与 CrossEntropy

In [35]:
import numpy as np

def kl_div(p, q, eps=1e-12):
    p = np.clip(p, eps, 1.0) # 防止 log(0)
    q = np.clip(q, eps, 1.0)
    return np.sum(p * (np.log(p) - np.log(q))) # 计算 KL 散度

def cross_entropy(p, q, eps=1e-12):
    q = np.clip(q, eps, 1.0)
    return -np.sum(p * np.log(q)) # 计算交叉熵

Q10 cosine similarity + topk 检索

In [36]:
import numpy as np

def cosine_topk(query, docs, k=5):
    q = query / (np.linalg.norm(query) + 1e-12) # 归一化查询向量
    d = docs / (np.linalg.norm(docs, axis=1, keepdims=True) + 1e-12) # 归一化文档向量
    score = d @ q
    idx = np.argpartition(-score, k)[:k] # 找到前 k 个索引
    idx = idx[np.argsort(-score[idx])] # 对前 k 个索引按分数排序
    return idx, score[idx] # 返回索引和对应的分数

## 深度学习基础组件

Q11 stable softmax

减去最大值防止指数溢出,softmax 不变性保证结果不变

In [37]:
import numpy as np

def stable_softmax(x, axis=-1):
    x = x - np.max(x, axis=axis, keepdims=True) # 防止溢出
    exp_x = np.exp(x)
    return exp_x / np.sum(exp_x, axis=axis, keepdims=True) # 计算 softmax

交叉熵损失的核心原理，是衡量模型预测的概率分布 $Q$ 与真实分布 $P$ 之间的差异。其定义为：

$$
H(P, Q) = -\sum_{x} P(x) \log Q(x)
$$

在分类任务中，真实标签通常为 one‑hot 向量（仅正确类别对应位置为 1，其余为 0），此时交叉熵简化为：

$$
\mathcal{L} = -\log p_{\text{true}}
$$

其中 $p_{\text{true}}$ 是模型对**真实类别**的预测概率。

**直观理解**：
- 预测越准确（$p_{\text{true}} \to 1$），损失越接近 0
- 预测越离谱（$p_{\text{true}} \to 0$），损失会急剧增大

**优化视角**：
最小化交叉熵等价于最大化真实类别的对数似然，即极大似然估计。配合 Softmax 使用时，梯度形式极为简洁：

$$
\frac{\partial \mathcal{L}}{\partial z_i} = p_i - y_i
$$

错误越大，梯度越大，避免了均方误差那样的饱和问题，因此优化过程更稳定高效。

In [38]:
import numpy as np

def ce_loss(logits, y):
    p = stable_softmax(logits, axis=1) # 计算 softmax 概率
    n = logits.shape[0]
    return -np.log(p[np.arange(n), y] + 1e-12).mean() # 计算交叉熵损失

BCEWithLogitsLoss = Sigmoid激活函数 + 二元交叉熵(BCELoss)

模型输入为未经Sigmoid映射的原始Logits，内部自动完成概率映射，具备更好的数值稳定性，适用于二分类、多标签分类任务。

$$
\mathcal{L} = -\frac{1}{N}\sum_{i=1}^{N}\Big[y_i \cdot \sigma(z_i) + (1-y_i)\cdot\big(1-\sigma(z_i)\big)\Big]
$$
$\sigma(z_i)=\dfrac{1}{1+e^{-z_i}}$ 为Sigmoid函数
$z_i$：模型原始输出Logits；$y_i\in[0,1]$：样本标签；$N$：样本总数

**PyTorch 内部稳定化简形式（避免显式Sigmoid，防止浮点溢出）**
$$
\ell_i = \max(z_i,0)-z_i y_i+\log\big(1+e^{-|z_i|}\big)
$$
$$
\mathcal{L} = \frac{1}{N}\sum_{i=1}^N \ell_i
$$

Q13 手写 BCEWithLogits loss

In [39]:
import numpy as np

def bce_with_logits(logits, y):
    x = logits
    return np.mean(np.maximum(x, 0) - x * y + np.log1p(np.exp(-np.abs(x)))) # 计算二分类交叉熵损失

Q14 手写 MSE loss + backward

In [40]:
def mse_loss(pred, target):
    diff = pred - target
    loss = np.mean(diff ** 2) # 计算均方误差
    grad = 2.0 * diff / diff.size
    
    return loss, grad

Q15 手写 ReLU / Sigmoid / GELU

In [41]:
import numpy as np

def relu(x):
    return np.maximum(0, x)

def sigmoid(x):
    return 1.0 / (1.0 + np.exp(-np.clip(x, -100, 100)))

def gelu(x):
    return 0.5 * x * (1 + np.tanh(np.sqrt(2 / np.pi) * (x + 0.044715 * x ** 3)))